In [88]:
#import libraries
import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score

import xgboost as xgb
import matplotlib.pyplot as plt
from sklearn.metrics import average_precision_score
from sklearn.metrics import roc_curve,auc, precision_recall_curve
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, precision_score

import warnings
warnings.filterwarnings("ignore")

In [89]:
DATA_PATH = '/Users/zamiulalam/Documents/Informatics_Project/mimic_processed_data/mimic3-benchmarks/data/in-hospital-mortality-7subseq-features/'

In [ ]:
train_df = pd.read_csv(DATA_PATH + "train_features.csv")
test_df  = pd.read_csv(DATA_PATH + "test_features.csv")

print(train_df.shape)
print(test_df.shape)

(17903, 883)
(3236, 883)


In [91]:
path = '/Users/zamiulalam/Documents/Informatics_Project/mimic_processed_data/mimic3-benchmarks/data/root/all_stays.csv'

stays = pd.read_csv(path)

In [92]:
train_df.head()

,Capillary refill rate_full_mean,Capillary refill rate_full_std,Capillary refill rate_full_min,Capillary refill rate_full_max,Capillary refill rate_full_skew,Capillary refill rate_full_count,Capillary refill rate_full_missing,Capillary refill rate_first10_mean,Capillary refill rate_first10_std,Capillary refill rate_first10_min,...,age_last25_count,age_last25_missing,age_last10_mean,age_last10_std,age_last10_min,age_last10_max,age_last10_skew,age_last10_count,age_last10_missing,label
0,NaN,NaN,NaN,NaN,NaN,0,1,NaN,NaN,NaN,...,12,0,83.839084,0.000000e+00,83.839084,83.839084,NaN,5,0,0
1,NaN,NaN,NaN,NaN,NaN,0,1,NaN,NaN,NaN,...,12,0,68.246644,0.000000e+00,68.246644,68.246644,NaN,5,0,0
2,NaN,NaN,NaN,NaN,NaN,0,1,NaN,NaN,NaN,...,12,0,62.977709,0.000000e+00,62.977709,62.977709,NaN,5,0,0
3,NaN,NaN,NaN,NaN,NaN,0,1,NaN,NaN,NaN,...,12,0,54.988710,7.944109e-15,54.988710,54.988710,NaN,5,0,0
4,NaN,NaN,NaN,NaN,NaN,0,1,NaN,NaN,NaN,...,12,0,71.672987,0.000000e+00,71.672987,71.672987,NaN,5,0,0


In [93]:
#separate features and labels
X_train = train_df.drop(columns=["label"])
y_train = train_df["label"]

X_test = test_df.drop(columns=["label"])
y_test = test_df["label"]

In [94]:
# Mean imputation
imputer = SimpleImputer(strategy="mean")

X_train = imputer.fit_transform(X_train)
X_test  = imputer.transform(X_test)

In [ ]:
#train XGBoost
from xgboost import XGBClassifier

model = xgb.XGBClassifier(
    n_estimators=2000,
    max_depth=8,
    learning_rate=0.02,
    subsample=0.8,
    colsample_bytree=0.5,
    eval_metric='aucpr',
    tree_method='hist',
    nthread=4,
)

model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.5, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='aucpr', feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.02, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=8,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=2000,
              n_jobs=None, nthread=4, ...)

In [96]:
#evaluation on test set
test_probs = model.predict_proba(X_test)[:,1]

print("Test ROC:", roc_auc_score(y_test, test_probs))
print("Test PR:", average_precision_score(y_test, test_probs))

Test ROC: 0.8791316793536548
Test PR: 0.5714573250433
